# SkillPulse — Setup & Data Collection
Run cells top to bottom. Make sure `.env` and MySQL are set up first (see README notes).

## 1. Imports

In [1]:
import json as pyjson

In [2]:
import os
import time
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv(dotenv_path="../.env")  # reads variables from .env in the same folder


True

## 2. Connect to MySQL
This reads credentials from `.env` — never hardcode them here.

In [3]:

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

#engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
# quick connection test
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))
    print("Connected. Tables found:", [row[0] for row in result])


Connected. Tables found: ['drift_reports', 'job_postings', 'job_skills', 'model_runs', 'skills']


## 3. Adzuna scraper function
Pulls one page of results at a time.

In [4]:
ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")

def fetch_page(page, country="in", keyword="data scientist", results_per_page=50):
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": results_per_page,
        "what": keyword,
        "content-type": "application/json",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json().get("results", [])


## 4. Test pull — small batch first
Check salary-field coverage before running the full 5,000–10,000 pull.

In [5]:
test_results = fetch_page(page=1, keyword="data scientist", results_per_page=50)
df_test = pd.json_normalize(test_results)
print("Rows pulled:", len(df_test))

for col in ["salary_min", "salary_max"]:
    if col not in df_test.columns:
        df_test[col] = pd.NA

print("Salary fields populated:", df_test[["salary_min", "salary_max"]].notna().mean())
df_test.head()

Rows pulled: 50
Salary fields populated: salary_min    0.04
salary_max    0.04
dtype: float64


,created,description,title,redirect_url,id,salary_is_predicted,longitude,adref,__CLASS__,latitude,...,category.__CLASS__,location.__CLASS__,location.area,location.display_name,company.display_name,company.__CLASS__,contract_time,contract_type,salary_max,salary_min
0,2026-07-18T18:10:14Z,Data Science Instructor (Jaipur) upGrad is exp...,Data Science Instructor,https://www.adzuna.in/land/ad/5806140450?se=eC...,5806140450,0,75.946900,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgwNjE0MDQ1MCIsI...,Adzuna::API::Response::Job,26.74410,...,Adzuna::API::Response::Category,Adzuna::API::Response::Location,"[India, Rajasthan, Jaipur]","Jaipur, Rajasthan",upGrad,Adzuna::API::Response::Company,NaN,NaN,NaN,NaN
1,2026-07-03T17:37:09Z,Data Science & AI Instructor upGrad is expandi...,Data Science Instructor,https://www.adzuna.in/land/ad/5786785503?se=eC...,5786785503,0,NaN,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc4Njc4NTUwMyIsI...,Adzuna::API::Response::Job,NaN,...,Adzuna::API::Response::Category,Adzuna::API::Response::Location,[India],India,upGrad,Adzuna::API::Response::Company,NaN,NaN,NaN,NaN
2,2026-07-18T18:07:28Z,Job Title: Data Science Instructor – Offline L...,Data Science Instructor,https://www.adzuna.in/land/ad/5806136048?se=eC...,5806136048,0,79.905350,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgwNjEzNjA0OCIsI...,Adzuna::API::Response::Job,23.13080,...,Adzuna::API::Response::Category,Adzuna::API::Response::Location,"[India, Madhya Pradesh, Jabalpur]","Jabalpur, Madhya Pradesh",upGrad,Adzuna::API::Response::Company,NaN,NaN,NaN,NaN
3,2026-07-03T17:35:51Z,Part-Time Data Science & AI Instructor (Chandi...,Part time Data Science Instructor,https://www.adzuna.in/land/ad/5786785222?se=eC...,5786785222,0,76.760940,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc4Njc4NTIyMiIsI...,Adzuna::API::Response::Job,30.75395,...,Adzuna::API::Response::Category,Adzuna::API::Response::Location,"[India, Chandigarh]","Chandigarh, India",upGrad,Adzuna::API::Response::Company,part_time,NaN,NaN,NaN
4,2026-07-04T17:07:19Z,Designation: Assistant Professor - Computer Sc...,Assistant Professor of Computer Science (Data ...,https://www.adzuna.in/land/ad/5787965226?se=eC...,5787965226,0,74.014575,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiZUNMUWh2Q0U4UkdiX...,Adzuna::API::Response::Job,15.35420,...,Adzuna::API::Response::Category,Adzuna::API::Response::Location,"[India, Goa]","Goa, India",Ganpat University Goa,Adzuna::API::Response::Company,NaN,NaN,NaN,NaN


In [6]:
import json
print(json.dumps(test_results[0], indent=2))

{
  "created": "2026-07-18T18:10:14Z",
  "category": {
    "tag": "it-jobs",
    "label": "IT Jobs",
    "__CLASS__": "Adzuna::API::Response::Category"
  },
  "description": "Data Science Instructor (Jaipur) upGrad is expanding its Offline Learning Centres , and we are looking for a dynamic Data Science Instructor in Jaipur Key Responsibilities As a Instructor, you will play a vital role in ensuring high-quality learning outcomes: Deliver High-Impact Sessions: Conduct engaging, in-person lectures and hands-on training sessions in Data Science, Machine Learning, Deep Learning, and cutting-edge GenAI/Agentic AI frameworks. Project Mentorship: Guide learners through re\u2026",
  "title": "Data Science Instructor",
  "redirect_url": "https://www.adzuna.in/land/ad/5806140450?se=eCLQhvCE8RGb_eC7-Bq01A&utm_medium=api&utm_source=7a91684d&v=6D2845F232523FDFE0FDD8CB7BC2306085E389FE",
  "id": "5806140450",
  "salary_is_predicted": "0",
  "location": {
    "__CLASS__": "Adzuna::API::Response::Loca

In [7]:
salary_count = sum(1 for r in test_results if r.get("salary_min") is not None)
print(f"Postings with salary_min present: {salary_count} out of {len(test_results)}")

Postings with salary_min present: 2 out of 50


In [8]:
# Test salary coverage across different countries
for country_code in ["gb", "us"]:
    results = fetch_page(page=1, keyword="data scientist", results_per_page=50, country=country_code)
    salary_count = sum(1 for r in results if r.get("salary_min") is not None)
    print(f"{country_code}: {salary_count} out of {len(results)} postings have salary_min")

gb: 50 out of 50 postings have salary_min
us: 50 out of 50 postings have salary_min


## 5. Insert into MySQL
Writes each result into the `job_postings` table.

In [9]:
def insert_postings(results, source="adzuna", country="gb"):
    rows = []
    for r in results:
        rows.append({
            "source": source,
            "country": country,
            "title": r.get("title"),
            "company": r.get("company", {}).get("display_name"),
            "location": r.get("location", {}).get("display_name"),
            "salary_min": r.get("salary_min"),
            "salary_max": r.get("salary_max"),
            "description": r.get("description"),
            "posted_date": r.get("created", "")[:10] or None,
            "raw_json": pyjson.dumps(r),
        })
    df = pd.DataFrame(rows)
    df.to_sql("job_postings", con=engine, if_exists="append", index=False)
    print(f"Inserted {len(df)} rows ({country})")

## 6. Full paginated pull
Loops through multiple pages with rate-limit delay. Adjust `max_pages` and `keywords` as needed.

In [10]:
countries = ["in", "gb", "us"]
keywords = ["data scientist", "software engineer", "machine learning engineer", "backend developer"]
max_pages = 10

for country_code in countries:
    for kw in keywords:
        for page in range(1, max_pages + 1):
            try:
                results = fetch_page(page=page, keyword=kw, results_per_page=50, country=country_code)
                if not results:
                    break
                insert_postings(results, source="adzuna", country=country_code)
                time.sleep(1)
            except Exception as e:
                print(f"Error on {country_code}/{kw} page {page}: {e}")
                time.sleep(5)

Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (in)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 50 rows (gb)
Inserted 5

In [11]:
with engine.connect() as conn:
    result = conn.execute(text("DELETE FROM job_postings WHERE country IS NULL"))
    conn.commit()
    print(f"Deleted {result.rowcount} rows with missing country")

Deleted 0 rows with missing country


## 7. Sanity check
Confirm what landed in the database.

In [12]:
df_check = pd.read_sql(
    "SELECT country, COUNT(*) AS total, SUM(salary_min IS NOT NULL) AS with_salary FROM job_postings GROUP BY country;",
    con=engine
)
df_check

,country,total,with_salary
0,in,16100,6997.0
1,gb,16000,15986.0
2,us,15550,15550.0
